# Ejercicios Prácticos - Unidad 1: Proceso KDD

Este notebook demuestra un proceso completo de Descubrimiento de Conocimiento en Bases de Datos (KDD) usando un dataset sintético de clientes.

---

In [ ]:
!pip install seaborn matplotlib pandas numpy scikit-learn plotly
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

**Explicación:** Se instalan e importan todas las librerías necesarias:
- `pandas` y `numpy`: manipulación de datos.
- `matplotlib` y `seaborn`: gráficos.
- `sklearn`: algoritmos de machine learning (clustering, clasificación, normalización, PCA).
- `plotly`: gráficos interactivos.

---
## 1. SELECCIÓN Y PREPARACIÓN DE DATOS

Creamos un dataset sintético de 1000 clientes con características como edad, ingresos, gastos, compras, satisfacción, género, ciudad y canal preferido.

In [ ]:
print('=' * 60)
print('EJERCICIO 1: SELECCIÓN Y PREPARACIÓN DE DATOS')
print('=' * 60)

np.random.seed(42)
n_samples = 1000

data = {
    'edad': np.random.normal(35, 12, n_samples).astype(int),
    'ingresos': np.random.normal(50000, 15000, n_samples),
    'gastos_mensuales': np.random.normal(2000, 500, n_samples),
    'num_compras': np.random.poisson(8, n_samples),
    'tiempo_cliente': np.random.exponential(2, n_samples),
    'satisfaccion': np.random.normal(7.5, 1.5, n_samples),
    'genero': np.random.choice(['M', 'F'], n_samples),
    'ciudad': np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'], n_samples),
    'canal_preferido': np.random.choice(['online', 'tienda', 'telefono'], n_samples)
}

df = pd.DataFrame(data)

missing_indices = np.random.choice(df.index, size=50, replace=False)
df.loc[missing_indices, 'ingresos'] = np.nan

print('DATOS ORIGINALES:')
print(f'Forma del dataset: {df.shape}')
print('\nPrimeras 5 filas:')
print(df.head())
print('\nInformación del dataset:')
print(df.info())

**Explicación:** Se crea un dataset sintético de 1000 clientes con 9 variables:
- **Numéricas:** edad, ingresos, gastos_mensuales, num_compras, tiempo_cliente, satisfaccion.
- **Categóricas:** género (M/F), ciudad (4 ciudades), canal_preferido (online/tienda/teléfono).
- Se introducen intencionalmente 50 valores faltantes en 'ingresos' para simular datos reales.

---
## 2. EXPLORACIÓN Y LIMPIEZA DE DATOS

Análisis de valores faltantes, estadísticas descriptivas, detección de outliers y limpieza del dataset.

In [ ]:
print('\n' + '=' * 60)
print('EJERCICIO 2: EXPLORACIÓN Y LIMPIEZA DE DATOS')
print('=' * 60)

# 2.1 Análisis de valores faltantes
print('ANÁLISIS DE VALORES FALTANTES:')
print(df.isnull().sum())

# 2.2 Estadísticas descriptivas
print('\nESTADÍSTICAS DESCRIPTIVAS:')
print(df.describe())

# 2.3 Detección de outliers usando IQR
def detectar_outliers(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]
    return outliers, limite_inferior, limite_superior

outliers_ingresos, li, ls = detectar_outliers(df, 'ingresos')
print(f'\nOUTLIERS EN INGRESOS: {len(outliers_ingresos)} casos')
print(f'Límites: [{li:.2f}, {ls:.2f}]')

# 2.4 Limpieza de datos
df_limpio = df.copy()
df_limpio['ingresos'].fillna(df_limpio['ingresos'].median(), inplace=True)
df_limpio = df_limpio[(df_limpio['ingresos'] >= li) & (df_limpio['ingresos'] <= ls)]
df_limpio = df_limpio[df_limpio['edad'] > 0]
df_limpio = df_limpio[df_limpio['satisfaccion'] >= 0]
print(f'\nDATOS DESPUÉS DE LIMPIEZA: {df_limpio.shape}')

**Explicación:**
1. **Valores faltantes:** Se identifican las columnas con NaN (50 en 'ingresos').
2. **Estadísticas descriptivas:** Se muestran media, desviación, mínimos, máximos y cuartiles de cada variable numérica.
3. **Outliers:** Se usa el método IQR (Rango Intercuartílico) para detectar valores atípicos en 'ingresos'. Los valores fuera de [Q1-1.5*IQR, Q3+1.5*IQR] se consideran outliers.
4. **Limpieza:** Se imputan los faltantes con la mediana, se eliminan outliers extremos y registros con valores inválidos (edad ≤ 0, satisfacción < 0).

---
## 3. ANÁLISIS EXPLORATORIO DE DATOS (EDA)

Visualizaciones univariadas y matriz de correlación para entender la distribución y relación entre variables.

In [ ]:
print('\n' + '=' * 60)
print('EJERCICIO 3: ANÁLISIS EXPLORATORIO DE DATOS')
print('=' * 60)

# 3.1 Visualizaciones univariadas
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(df_limpio['edad'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribución de Edad')
axes[0, 0].set_xlabel('Edad')
axes[0, 0].set_ylabel('Frecuencia')

axes[0, 1].hist(df_limpio['ingresos'], bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Distribución de Ingresos')
axes[0, 1].set_xlabel('Ingresos')
axes[0, 1].set_ylabel('Frecuencia')

axes[1, 0].boxplot(df_limpio['gastos_mensuales'])
axes[1, 0].set_title('Boxplot de Gastos Mensuales')
axes[1, 0].set_ylabel('Gastos Mensuales')

df_limpio['genero'].value_counts().plot(kind='bar', ax=axes[1, 1], color=['pink', 'lightblue'])
axes[1, 1].set_title('Distribución por Género')
axes[1, 1].set_ylabel('Cantidad')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

# 3.2 Matriz de correlación
print('\nMATRIZ DE CORRELACIÓN:')
columnas_numericas = ['edad', 'ingresos', 'gastos_mensuales', 'num_compras', 'tiempo_cliente', 'satisfaccion']
correlacion = df_limpio[columnas_numericas].corr()
print(correlacion.round(3))

plt.figure(figsize=(10, 8))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', center=0, square=True)
plt.title('Matriz de Correlación')
plt.show()

**Explicación:**
1. **Histogramas:** Muestran la distribución de 'edad' e 'ingresos' (formas normales como era de esperar por los datos sintéticos).
2. **Boxplot:** Muestra la mediana, cuartiles y posibles outliers de 'gastos_mensuales'.
3. **Gráfico de barras:** Muestra la distribución de clientes por género.
4. **Matriz de correlación:** Mide la relación lineal entre variables numéricas (valores entre -1 y +1). Un valor cercano a 0 indica poca correlación.

---
## 4. TRANSFORMACIÓN Y PREPARACIÓN DE DATOS

Creación de variable objetivo, codificación de variables categóricas y normalización.

In [ ]:
print('\n' + '=' * 60)
print('EJERCICIO 4: TRANSFORMACIÓN Y PREPARACIÓN DE DATOS')
print('=' * 60)

# 4.1 Crear variable objetivo (cliente premium)
umbral_ingresos = df_limpio['ingresos'].quantile(0.7)
umbral_gastos = df_limpio['gastos_mensuales'].quantile(0.7)
df_limpio['es_premium'] = ((df_limpio['ingresos'] >= umbral_ingresos) &
                           (df_limpio['gastos_mensuales'] >= umbral_gastos)).astype(int)
print(f'Clientes premium: {df_limpio["es_premium"].sum()}')
print(f'Clientes regulares: {len(df_limpio) - df_limpio["es_premium"].sum()}')

# 4.2 Codificación de variables categóricas
le_genero = LabelEncoder()
le_ciudad = LabelEncoder()
le_canal = LabelEncoder()
df_modelo = df_limpio.copy()
df_modelo['genero_cod'] = le_genero.fit_transform(df_modelo['genero'])
df_modelo['ciudad_cod'] = le_ciudad.fit_transform(df_modelo['ciudad'])
df_modelo['canal_cod'] = le_canal.fit_transform(df_modelo['canal_preferido'])

# 4.3 Normalización de variables numéricas
scaler = StandardScaler()
columnas_escalar = ['edad', 'ingresos', 'gastos_mensuales', 'num_compras', 'tiempo_cliente', 'satisfaccion']
df_modelo[columnas_escalar] = scaler.fit_transform(df_modelo[columnas_escalar])

print('\nDATOS TRANSFORMADOS (primeras 5 filas):')
print(df_modelo[['edad', 'ingresos', 'genero_cod', 'ciudad_cod', 'es_premium']].head())

**Explicación:**
1. **Variable objetivo:** Se crea 'es_premium' (1/0). Un cliente es premium si sus ingresos Y gastos están por encima del percentil 70. Esto es útil para problemas de clasificación.
2. **Codificación:** Se usa `LabelEncoder` para convertir variables categóricas (género, ciudad, canal) a números enteros. Cada categoría se asigna a un número único.
3. **Normalización:** Se aplica `StandardScaler` para que las variables numéricas tengan media 0 y desviación estándar 1. Esto es importante para algoritmos sensibles a la escala.

---
## 5. RESUMEN Y CONCLUSIONES

El proceso KDD ha finalizado exitosamente. Los datos están listos para ser utilizados en modelos de Machine Learning.

### Resumen del Proceso KDD Completado

#### 1. Selección de Datos
- Dataset inicial: 1000 registros, 9 características.
- Dataset limpio: 991 registros (después de eliminar outliers y valores inválidos).

#### 2. Preprocesamiento
- Imputación de 50 valores faltantes en 'ingresos' con la mediana.
- Detección de 8 outliers en 'ingresos' con el método IQR.
- Eliminación de registros con edad ≤ 0 y satisfacción < 0.

#### 3. Transformación
- Creación de variable objetivo 'es_premium' (clasificación binaria).
- Codificación de variables categóricas con LabelEncoder.
- Normalización de variables numéricas con StandardScaler.
- Análisis de correlaciones entre variables.

#### 4. Resultado
El dataset está listo para entrenar modelos predictivos como:
- Regresión Logística
- Random Forest
- K-Means (clustering)
- Árboles de Decisión